# WEEK-08: K-Nearest Neighbour (K-NN) & ID-3 Decision Tree

Lab implementation of Q-1.A/B, Q-2.A/B and Additional Questions 1–3.

- From-scratch KNN uses Euclidean / Manhattan / Minkowski
- From-scratch ID3 uses Entropy and Information Gain
- `.B` sections use `scikit-learn`

In [2]:
import numpy as np
import pandas as pd
from collections import Counter
import math
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 6)

## Shared utilities — distance + scratch KNN

In [3]:
def distance(x1, x2, mode="euclid", p=3):
    x1, x2 = np.asarray(x1, dtype=float), np.asarray(x2, dtype=float)
    if mode in ("euclid", "euclidean"):
        return np.linalg.norm(x1 - x2, ord=2)
    if mode in ("manhattan", "manhatten"):
        return np.linalg.norm(x1 - x2, ord=1)
    if mode in ("minkowski",):
        return np.linalg.norm(x1 - x2, ord=p)
    raise ValueError(f"Unknown mode: {mode}")


def knn_predict_one(X_train, y_train, x_query, k=3, mode="euclid", p=3):
    dists = np.array([distance(x_query, x, mode, p) for x in X_train])
    nn_idx = np.argsort(dists)[:k]
    nn_labels = np.asarray(y_train)[nn_idx]
    pred = Counter(nn_labels).most_common(1)[0][0]
    return pred, dists, nn_idx


def knn_predict(X_train, y_train, X_test, k=3, mode="euclid", p=3):
    X_test = np.atleast_2d(np.asarray(X_test, dtype=float))
    return np.array([knn_predict_one(X_train, y_train, x, k, mode, p)[0] for x in X_test])


def all_distances_table(X, names, query, p=3):
    rows = []
    for i, x in enumerate(X):
        rows.append({
            "sample": names[i],
            "euclidean": distance(query, x, "euclid"),
            "manhattan": distance(query, x, "manhattan"),
            "minkowski_p3": distance(query, x, "minkowski", p=p),
        })
    return pd.DataFrame(rows).sort_values("euclidean")


def plot_knn_2d(X, y, query=None, k=3, mode="euclid",
                label_colors=None, title="", query_label="query",
                xlabel="x1", ylabel="x2"):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    classes = list(dict.fromkeys(y))
    if label_colors is None:
        cmap = plt.cm.get_cmap("tab10", len(classes))
        label_colors = {c: cmap(i) for i, c in enumerate(classes)}

    pad_x = max(1.0, 0.1 * (X[:, 0].max() - X[:, 0].min() + 1e-9))
    pad_y = max(1.0, 0.1 * (X[:, 1].max() - X[:, 1].min() + 1e-9))
    x_min, x_max = X[:, 0].min() - pad_x, X[:, 0].max() + pad_x
    y_min, y_max = X[:, 1].min() - pad_y, X[:, 1].max() + pad_y
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 180),
                         np.linspace(y_min, y_max, 180))
    grid = np.c_[xx.ravel(), yy.ravel()]
    class_to_int = {c: i for i, c in enumerate(classes)}
    Z = np.array([class_to_int[knn_predict_one(X, y, pt, k, mode)[0]] for pt in grid])
    Z = Z.reshape(xx.shape)

    plt.figure()
    plt.contourf(xx, yy, Z, alpha=0.25, levels=np.arange(len(classes) + 1) - 0.5)
    for c in classes:
        pts = X[y == c]
        plt.scatter(pts[:, 0], pts[:, 1], c=[label_colors[c]],
                    edgecolors="k", s=80, label=str(c))
    if query is not None:
        q = np.asarray(query, dtype=float)
        plt.scatter(q[0], q[1], c="black", marker="*", s=220,
                    edgecolors="white", label=query_label, zorder=5)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

---
# Q-1.A Classification of Fruits (from scratch)

Features: Weight (g), Sweetness (1–10). Labels: Apple / Orange.

Classify a new fruit **(165 g, sweetness 5.5)** with **k = 3**.

In [ ]:
fruits = pd.DataFrame({
    "Weight":    [180, 200, 150, 170, 160, 140],
    "Sweetness": [7, 6, 4, 5, 6, 3],
    "Label":     ["Apple", "Apple", "Orange", "Orange", "Apple", "Orange"],
})
display(fruits)

X_fruit = fruits[["Weight", "Sweetness"]].to_numpy(float)
y_fruit = fruits["Label"].to_numpy()
q_fruit = np.array([165.0, 5.5])
fruit_names = [f"#{i+1} {lab}" for i, lab in enumerate(y_fruit)]

### Distances: Euclidean, Manhattan, Minkowski (p=3)

In [ ]:
dist_fruit = all_distances_table(X_fruit, fruit_names, q_fruit)
display(dist_fruit)

### Predict for k = 1, 3, 5

In [ ]:
for k in (1, 3, 5):
    pred, _, nn = knn_predict_one(X_fruit, y_fruit, q_fruit, k=k, mode="euclid")
    print(f"k={k}  neighbors={list(y_fruit[nn])}  ->  {pred}")

### Function-style KNN (last column = y, rest = X)

In [ ]:
def KNN(df, k=3, mode="euclid"):
    df = np.asarray(df)
    X, y = df[:, :-1].astype(float), df[:, -1]
    def predict(X_test):
        return knn_predict(X, y, X_test, k=k, mode=mode)
    return predict

raw = fruits[["Weight", "Sweetness", "Label"]].to_numpy()
predict = KNN(raw, k=3, mode="euclid")
print("KNN() prediction:", predict(q_fruit)[0])

### Plot: Apple = red, Orange = orange, plus decision boundary

In [ ]:
plot_knn_2d(
    X_fruit, y_fruit, q_fruit, k=3, mode="euclid",
    label_colors={"Apple": "red", "Orange": "orange"},
    title="Q-1.A Fruits  k=3  Euclidean",
    query_label="new fruit (165, 5.5)",
    xlabel="Weight (g)", ylabel="Sweetness",
)

**Effect of k**

- `k=1`: nearest point is Orange (170, 5) → **Orange**
- `k=3`: 2 Orange + 1 Apple → **Orange**
- `k=5`: 3 Apple + 2 Orange → **Apple**

Small `k` is jumpy; larger `k` smooths but can flip the vote.

---
# Q-1.B Fruits with scikit-learn

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import pairwise_distances

for metric, kwargs in [("euclidean", {}), ("manhattan", {}), ("minkowski", {"p": 3})]:
    D = pairwise_distances(X_fruit, q_fruit.reshape(1, -1), metric=metric, **kwargs).ravel()
    print(metric, np.round(D, 3))

clf = KNeighborsClassifier(n_neighbors=3, metric="euclidean")
clf.fit(X_fruit, y_fruit)
print("sklearn k=3:", clf.predict(q_fruit.reshape(1, -1))[0])

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y_fruit)
clf_plot = KNeighborsClassifier(n_neighbors=3, metric="euclidean")
clf_plot.fit(X_fruit, y_enc)

fig, ax = plt.subplots()
DecisionBoundaryDisplay.from_estimator(
    clf_plot, X_fruit, response_method="predict", alpha=0.25, ax=ax
)
for lab, color in [("Apple", "red"), ("Orange", "orange")]:
    pts = X_fruit[y_fruit == lab]
    ax.scatter(pts[:, 0], pts[:, 1], c=color, edgecolors="k", s=80, label=lab)
ax.scatter(q_fruit[0], q_fruit[1], c="black", marker="*", s=220, label="query")
ax.set_title("Q-1.B sklearn KNN decision boundary")
ax.legend()
plt.show()